# Task 6 — Three-Layer Analysis

Loads pre-computed results from `src/analysis/task6_compute.py` and renders all three
evaluation layers across all run/dataset combos.

- **Layer 1**: Standard metrics (precision / recall / F1)
- **Layer 2**: Augmented dataset characteristics (ratio, corner-case %, attribute coverage,
  overlap between strategies)
- **Layer 3**: Qualitative error analysis (error class breakdown + LLM-explained examples)

To re-run computations: `python src/analysis/task6_compute.py`

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "experiments" / "results"

with open(RESULTS / "task6_dataset_characteristics.json") as f:
    chars = json.load(f)
with open(RESULTS / "task6_error_analysis.json") as f:
    errors = json.load(f)

layer1 = chars["layer1"]
layer2 = chars["layer2"]

RUNS = {
    "wdc-products":  ["baseline_cw", "string_aug_cw", "llm_aug_cw", "web_aug_cw"],
    "dblp-scholar":  ["baseline",    "string_aug",    "llm_aug",    "web_aug"],
}
STRATEGIES = ["baseline", "string_aug", "llm_aug", "web_aug"]
print("Data loaded.")

Data loaded.


## Layer 1 — Standard Metrics

All 8 runs across both datasets.

In [2]:
rows = []
for dataset, runs in RUNS.items():
    for run in runs:
        d = layer1[dataset][run]
        m = d["metrics"]
        cm = d["confusion_matrix"]
        correct = cm["tp"] + cm["tn"]
        n = d["n_pairs"]
        rows.append({
            "Dataset": dataset,
            "Run": run,
            "Precision": round(m["precision"], 4),
            "Recall":    round(m["recall"],    4),
            "F1":        round(m["f1"],        4),
            "TP": cm["tp"], "FP": cm["fp"], "FN": cm["fn"], "TN": cm["tn"],
            "Accuracy": round(correct / n, 4),
        })
df1 = pd.DataFrame(rows)
df1

,Dataset,Run,Precision,Recall,F1,TP,FP,FN,TN,Accuracy
0,wdc-products,baseline_cw,0.5587,0.7520,0.6411,376,297,124,3703,0.9064
1,wdc-products,string_aug_cw,0.6506,0.7040,0.6763,352,189,148,3811,0.9251
2,wdc-products,llm_aug_cw,0.5099,0.7720,0.6142,386,371,114,3629,0.8922
3,wdc-products,web_aug_cw,0.5963,0.7180,0.6515,359,243,141,3757,0.9147
4,dblp-scholar,baseline,0.9487,0.9673,0.9579,1035,56,35,4616,0.9842
5,dblp-scholar,string_aug,0.9384,0.9533,0.9458,1020,67,50,4605,0.9796
6,dblp-scholar,llm_aug,0.9498,0.9542,0.9520,1021,54,49,4618,0.9821
7,dblp-scholar,web_aug,0.9479,0.9523,0.9501,1019,56,51,4616,0.9814


### Z-test significance (accuracy-proxy, α=0.05)

| Comparison | p-value | Significant? |
|---|---|---|
| WDC `web_aug_cw` vs `baseline_cw` | 0.017 | ✅ (worse) |
| WDC `web_aug_cw` vs `string_aug_cw` (best) | <0.001 | ✅ (worse) |
| WDC `string_aug_cw` vs `baseline_cw` | 0.0014 | ✅ (better) |
| WDC `llm_aug_cw` vs `baseline_cw` | 0.828 | ❌ |
| DBLP `web_aug` vs `baseline` | 0.425 | ❌ |
| DBLP `web_aug` vs `llm_aug` | 0.373 | ❌ |
| DBLP `web_aug` vs `string_aug` | 0.306 | ❌ |

**Winner per dataset**: WDC → `string_aug_cw` (F1=0.676, only significant improvement).
DBLP → `baseline` remains best (all augmentations statistically indistinguishable).

## Layer 2 — Dataset Characteristics

### 2a. Pos/neg ratio & corner-case proportion (all strategies × datasets)

In [3]:
rows = []
for dataset in ["wdc-products", "dblp-scholar"]:
    for strat in STRATEGIES:
        s = layer2[dataset].get(strat, {})
        rows.append({
            "Dataset":    dataset,
            "Strategy":   strat,
            "File":       s.get("file", ""),
            "N":          s.get("n", ""),
            "% Positive": s.get("pct_pos", ""),
            "Hard Pos":   s.get("hard_pos", ""),
            "Hard Neg":   s.get("hard_neg", ""),
            "% Corner":   s.get("pct_corner", ""),
        })
df2 = pd.DataFrame(rows)
df2

,Dataset,Strategy,File,N,% Positive,Hard Pos,Hard Neg,% Corner
0,wdc-products,baseline,train.txt,2500,20.0,435,65,20.0
1,wdc-products,string_aug,train_aug_string.txt,5000,20.0,882,123,20.1
2,wdc-products,llm_aug,train_aug_llm.txt,4600,15.6,595,211,17.5
3,wdc-products,web_aug,train_aug_web.txt,3303,34.8,980,66,31.7
4,dblp-scholar,baseline,train.txt,17223,18.6,200,344,3.2
5,dblp-scholar,string_aug,train_aug_string.txt,34446,18.6,935,567,4.4
6,dblp-scholar,llm_aug,train_aug_llm.txt,19308,16.6,200,385,3.0
7,dblp-scholar,web_aug,train_aug_web.txt,17965,21.4,337,351,3.8


### 2b. Decisive-attribute coverage (WDC Products only)

In [4]:
rows = []
for strat in STRATEGIES:
    s = layer2["wdc-products"].get(strat, {})
    row = {"Strategy": strat}
    for attr, d in s.get("attr_coverage", {}).items():
        row[attr] = f"{d['pct']}%"
    rows.append(row)
pd.DataFrame(rows)

,Strategy,color,memory/storage,size,bundle,edition,model number
0,baseline,36.4%,40.5%,36.8%,13.2%,20.7%,35.0%
1,string_aug,35.5%,39.9%,36.4%,13.0%,20.3%,34.1%
2,llm_aug,36.9%,35.2%,35.7%,13.0%,20.7%,31.8%
3,web_aug,35.0%,37.9%,34.9%,13.0%,19.4%,33.2%


### 2c. Overlap between augmentation strategies' added pairs

In [5]:
for dataset in ["wdc-products", "dblp-scholar"]:
    print(f"\n{dataset}")
    counts = layer2[dataset]["_added_pair_counts"]
    print("  Added pair counts:", counts)
    print("  Pairwise Jaccard overlap between added sets:")
    for pair, v in layer2[dataset]["_overlap"].items():
        print(f"    {pair}: Jaccard={v['jaccard']} ({v['intersection']} common / {v['union']} union)")
print("\nZero overlap across all pairs and datasets — each strategy explores a completely disjoint")
print("region of pair space, confirming complementarity rather than redundancy.")


wdc-products
  Added pair counts: {'string_aug': 2500, 'llm_aug': 2100, 'web_aug': 800}
  Pairwise Jaccard overlap between added sets:
    string_aug_vs_llm_aug: Jaccard=0.0 (0 common / 4600 union)
    string_aug_vs_web_aug: Jaccard=0.0 (0 common / 3300 union)
    llm_aug_vs_web_aug: Jaccard=0.0 (0 common / 2900 union)

dblp-scholar
  Added pair counts: {'string_aug': 17223, 'llm_aug': 2031, 'web_aug': 710}
  Pairwise Jaccard overlap between added sets:
    string_aug_vs_llm_aug: Jaccard=0.0 (0 common / 19254 union)
    string_aug_vs_web_aug: Jaccard=0.0 (0 common / 17933 union)
    llm_aug_vs_web_aug: Jaccard=0.0 (0 common / 2741 union)

Zero overlap across all pairs and datasets — each strategy explores a completely disjoint
region of pair space, confirming complementarity rather than redundancy.


## Layer 3 — Qualitative Error Analysis

### 3a. Error class breakdown per run

In [6]:
rows = []
for key, d in errors.items():
    row = {
        "Run":     d["run"],
        "Dataset": d["dataset"],
        "Errors":  d["n_errors"],
        "% Error": round(100 * d["n_errors"] / d["n_total"], 1),
    }
    for cls in ["ambiguous_variant", "low_sim_match", "high_sim_non_match", "noisy_incomplete"]:
        row[cls] = d["class_counts"].get(cls, 0)
    rows.append(row)
df3 = pd.DataFrame(rows)
df3

,Run,Dataset,Errors,% Error,ambiguous_variant,low_sim_match,high_sim_non_match,noisy_incomplete
0,baseline_cw,wdc-products,421,9.4,299,103,16,3
1,string_aug_cw,wdc-products,337,7.5,208,122,5,2
2,llm_aug_cw,wdc-products,485,10.8,380,93,11,1
3,web_aug_cw,wdc-products,384,8.5,256,117,9,2
4,baseline,dblp-scholar,91,1.6,75,3,7,6
5,string_aug,dblp-scholar,117,2.0,101,3,10,3
6,llm_aug,dblp-scholar,103,1.8,86,2,11,4
7,web_aug,dblp-scholar,107,1.9,94,2,10,1


### 3b. Representative misclassification examples with LLM explanations

Change `SHOW_RUN` and `SHOW_DATASET` to inspect any run.

In [7]:
SHOW_RUN     = "string_aug_cw"   # change to inspect any run
SHOW_DATASET = "wdc-products"    # "wdc-products" | "dblp-scholar"

key = f"{SHOW_RUN}_{SHOW_DATASET}"
d = errors[key]
print(f"=== {key}  ({d['n_errors']} total errors) ===\n")
for cls, examples in d["representative_examples"].items():
    print(f"{'='*70}")
    print(f"CLASS: {cls.upper()}  (n={d['class_counts'].get(cls,0)})")
    print(f"{'='*70}")
    for i, ex in enumerate(examples, 1):
        print(f"\n  [{i}] true={'match' if ex['true_label']==1 else 'NON-MATCH'}"
              f"  pred={'MATCH' if ex['pred_label']==1 else 'non-match'}"
              f"  sim={ex['sim']:.3f}  score={ex['score']:.3f}")
        print(f"  LEFT : {ex['left'][:250]}")
        print(f"  RIGHT: {ex['right'][:250]}")
        print(f"  LLM  : {ex['llm_explanation']}")

=== string_aug_cw_wdc-products  (337 total errors) ===

CLASS: AMBIGUOUS_VARIANT  (n=208)

  [1] true=match  pred=non-match  sim=0.261  score=0.000
  LEFT : COL brand VAL  COL title VAL 3M - Privacy filter framed lightweight 20\"\" to 23\"\" widescreen COL description VAL 3M Privacy filter framed lightweight 20\"\" to 23\"\" widescreen (7000059523) - Type: Skjermbeskyttelse COL price VAL 1506.25 COL pric
  RIGHT: COL brand VAL  COL title VAL 3M - DESKTOP LCD LW FRAMED FILTER F-FEEDS COL description VAL 3M DESKTOP LCD LW FRAMED FILTER F-FEEDS (PF220W1F) - Type: Annet Tilbehør COL price VAL 3499.00 COL priceCurrency VAL NOK
  LLM  : The model likely predicted non-match because the RIGHT product uses an abbreviated, cryptic title ("DESKTOP LCD LW FRAMED FILTER F-FEEDS") that obscures its identity as the same 3M framed lightweight privacy filter for 20"-23" widescreen screens, while the large price discrepancy (1506 vs 3499 NOK) and differing product type labels further masked the semantic

### 3c. Full error example table (all runs, one class at a time)

In [8]:
SHOW_CLASS = "high_sim_non_match"  # ambiguous_variant | low_sim_match | high_sim_non_match | noisy_incomplete

print(f"=== Examples of class: {SHOW_CLASS} ===\n")
for key, d in errors.items():
    examples = d["representative_examples"].get(SHOW_CLASS, [])
    if not examples:
        continue
    print(f"--- {key} ({d['class_counts'].get(SHOW_CLASS,0)} total) ---")
    for ex in examples[:2]:  # print first 2 per run
        print(f"  sim={ex['sim']:.3f} | {ex['llm_explanation']}")
    print()

=== Examples of class: high_sim_non_match ===

--- baseline_cw_wdc-products (16 total) ---
  sim=0.857 | The model likely failed to distinguish between the subtle difference in model numbers **DS-7208HUHI-K2/P** (5MP, HUHI) versus **DS-7208HQHI-K2/P** (HQHI), which represent different product specifications, as the overwhelming similarity of all other tokens (brand, series, and feature descriptions) dominated the matching signal.
  sim=0.783 | The model likely overlooked the differing product specifications in the titles ("Ball B Length" vs "C Size") and the significant price difference (28 vs 43 EUR), which together indicate these are distinct variants of the RAM® Double Socket Arm rather than the same product.

--- string_aug_cw_wdc-products (5 total) ---
  sim=0.522 | The model likely focused on the shared brand name, description text, and "High Roller II" title while overlooking the critical size difference between the LEFT record (27.5 x **2.30**) and the RIGHT record (27.5 x **2.